# Decision tree and linear SVM

Nipun Yonjan

I am comparing two models on the shared phishing URL data:

1. A decision tree that stops at depth 8
2. A linear SVM

Both are fit on train and scored on dev. I am not using the test split.


## 1. Load the split

prepare_datasets() drops duplicate URLs and returns the same train, dev, and test split for the team. The pipeline fills in numeric columns with the median, standardizes them, and drops the raw url, dom, and tld text.

I fit the pipeline on train only, then transform dev. That way the scaler never sees dev or test.


In [1]:
from dataset_preparation import prepare_datasets

data = prepare_datasets()
pipeline = data["pipeline"]

X_train = pipeline.fit_transform(data["X_train"])
X_dev = pipeline.transform(data["X_dev"])
y_train = data["y_train"]
y_dev = data["y_dev"]

print("Prepared features:", X_train.shape[1])
print("Train rows:", X_train.shape[0])
print("Dev rows:", X_dev.shape[0])
print("Train phishing rate: {:.2%}".format(y_train.mean()))
print("Dev phishing rate:   {:.2%}".format(y_dev.mean()))
print("Feature names:", list(pipeline.get_feature_names_out()))

Prepared features: 22
Train rows: 69138
Dev rows: 23046
Train phishing rate: 14.40%
Dev phishing rate:   14.40%
Feature names: ['url_len', 'dom_len', 'is_ip', 'tld_len', 'subdom_cnt', 'letter_cnt', 'digit_cnt', 'special_cnt', 'eq_cnt', 'qm_cnt', 'amp_cnt', 'dot_cnt', 'dash_cnt', 'under_cnt', 'letter_ratio', 'digit_ratio', 'spec_ratio', 'is_https', 'slash_cnt', 'entropy', 'path_len', 'query_len']


After prep there are 22 numeric predictors. About 14% of train and about 14% of dev are phishing, so the split kept the balance. Test is still unused.


## 2. Decision tree

This tree uses entropy, stops at depth 8, needs 50 samples to split and 20 in each leaf, looks at sqrt of the features, and uses balanced class weights. That is different from Sarab's unlimited Gini tree.

Fit on train only. Phishing is label 1. ROC AUC uses the predicted phishing probability.


In [2]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

decision_tree = DecisionTreeClassifier(
    criterion="entropy",
    splitter="best",
    max_depth=8,
    min_samples_split=50,
    min_samples_leaf=20,
    max_features="sqrt",
    class_weight="balanced",
    random_state=42,
)
decision_tree.fit(X_train, y_train)

tree_pred = decision_tree.predict(X_dev)
tree_score = decision_tree.predict_proba(X_dev)[:, 1]

tree_metrics = {
    "accuracy": accuracy_score(y_dev, tree_pred),
    "precision": precision_score(y_dev, tree_pred, pos_label=1),
    "recall": recall_score(y_dev, tree_pred, pos_label=1),
    "f1": f1_score(y_dev, tree_pred, pos_label=1),
    "roc_auc": roc_auc_score(y_dev, tree_score),
}

print("Decision tree dev scores")
print("Accuracy:           {:.4f}".format(tree_metrics["accuracy"]))
print("Phishing precision: {:.4f}".format(tree_metrics["precision"]))
print("Phishing recall:    {:.4f}".format(tree_metrics["recall"]))
print("Phishing F1:        {:.4f}".format(tree_metrics["f1"]))
print("ROC AUC:            {:.4f}".format(tree_metrics["roc_auc"]))

Decision tree dev scores
Accuracy:           0.8970
Phishing precision: 0.5912
Phishing recall:    0.9228
Phishing F1:        0.7207
ROC AUC:            0.9670


On dev the tree is about 89.70% accuracy and 0.967 ROC AUC. It finds 92.28% of the phishing URLs, but only 59.12% of the URLs it flags are actually phishing. Phishing F1 is 0.721.

The balanced weights are pushing it to catch phishing instead of protecting accuracy. That recall is useful, since missing a phishing URL is worse than a false alarm. Precision is the weak part.


## 3. Linear SVM

This is a LinearSVC with C=0.01 and balanced class weights. Saurya's SVM uses C=1.0 and no class weight, so we are not running the same model. Penalty, dual=False, and max_iter match his.

LinearSVC has no predict_proba, so ROC AUC uses decision_function. Fit on train, scored on dev. Phishing is label 1.


In [3]:
from sklearn.svm import LinearSVC

linear_svm = LinearSVC(
    penalty="l2",
    loss="squared_hinge",
    dual=False,
    C=0.01,
    class_weight="balanced",
    max_iter=5000,
    random_state=42,
)
linear_svm.fit(X_train, y_train)

svm_pred = linear_svm.predict(X_dev)
svm_score = linear_svm.decision_function(X_dev)

svm_metrics = {
    "accuracy": accuracy_score(y_dev, svm_pred),
    "precision": precision_score(y_dev, svm_pred, pos_label=1),
    "recall": recall_score(y_dev, svm_pred, pos_label=1),
    "f1": f1_score(y_dev, svm_pred, pos_label=1),
    "roc_auc": roc_auc_score(y_dev, svm_score),
}

print("Linear SVM dev scores")
print("Accuracy:           {:.4f}".format(svm_metrics["accuracy"]))
print("Phishing precision: {:.4f}".format(svm_metrics["precision"]))
print("Phishing recall:    {:.4f}".format(svm_metrics["recall"]))
print("Phishing F1:        {:.4f}".format(svm_metrics["f1"]))
print("ROC AUC:            {:.4f}".format(svm_metrics["roc_auc"]))

Linear SVM dev scores
Accuracy:           0.9226
Phishing precision: 0.6669
Phishing recall:    0.9244
Phishing F1:        0.7748
ROC AUC:            0.9757


On dev the SVM is about 92.26% accuracy and 0.976 ROC AUC. It finds 92.44% of the phishing URLs, and 66.69% of the URLs it flags are actually phishing. Phishing F1 is 0.775.

It beats the tree on every dev metric. Both catch about 92% of the phishing URLs. The SVM just raises fewer false alarms.


## 4. Side by side

Same dev split for both. Phishing is label 1. Test is not in here.


In [4]:
import pandas as pd

comparison = pd.DataFrame(
    {
        "Decision tree": tree_metrics,
        "Linear SVM": svm_metrics,
    }
).T
comparison = comparison[["accuracy", "precision", "recall", "f1", "roc_auc"]]
comparison = comparison.rename(
    columns={
        "precision": "phishing_precision",
        "recall": "phishing_recall",
        "f1": "phishing_f1",
    }
)
comparison.round(4)

,accuracy,phishing_precision,phishing_recall,phishing_f1,roc_auc
Decision tree,0.8970,0.5912,0.9228,0.7207,0.9670
Linear SVM,0.9226,0.6669,0.9244,0.7748,0.9757


## Conclusion

The linear SVM looks stronger on dev. It is higher on accuracy, precision, recall, F1, and ROC AUC.

Recall is basically a tie. The tree finds 92.28% of phishing URLs and the SVM finds 92.44%. The SVM is just more precise, so its F1 is 0.775 against the tree's 0.721, and its ROC AUC is 0.976 against 0.967.

Accuracy is the least useful number here. About 86% of the URLs are benign, so a model can look fine and still miss phishing sites. I would look at phishing recall, F1, and ROC AUC first. On those, the SVM is the one I would take into the group's top three.
